# CollateralsREAll

Cloud migration of on-prem stored procedure `[bi].[sp_CollateralsRE]` which populates `[Analytic].[bi].[CollateralsREAll]`.

Produces one row per **Real-Estate collateral (mortgage) x property/ownership x agreement main-beneficiary**, delivered daily to an external company. The on-prem schema and values are preserved as closely as possible; see the **Known Differences** cell for documented gaps.

## Table mapping (on-prem -> cloud)

Primary source: `{env}-cdh.primaryactive.collateral` filtered to Real Estate (`Category = 'Επί ακινήτων'`).

| On-prem column | Cloud source |
| --- | --- |
| `PropertyKey` | `realestatebuilding.RealEstateCepalKey` (e.g. `CH230780`), joined via `collateral.RealEstateKey` |
| `CDI` (property owner) | `dparty.CDI` via `collateral.OwnerPartyKey = dparty.PartyKeyThesis`; fallback `NoCDI - PropertyOwnerId = <OwnershipKey>` |
| `AgreementNumber` | `collateral.AgreementKey` |
| `IntialLegalOrder` | `collateral.InitialLegalOrder` |
| `Currency` | `collateral.CollateralCurrency` |
| `CollateralAmountEuro` | `collateral.CollateralAmount` when EUR else NULL (on-prem convertToEuro returns NULL for non-EUR) |
| `CollateralAmount` | `collateral.CollateralAmount` |
| `primaryCDI` (agreement main beneficiary) | `exposure` Primary Owner CDI joined on `AgreementKey`; fallback `NoCDI - PropertyOwnerId = <OwnershipKey>` |
| `CollateralKey` | `collateral.CollateralOldKey` (e.g. `GLBCH424956`) |
| `ThesisCollateralID` | `collateral.CollateralKey` (bare number, e.g. `424956`) |
| `PortfolioId` | `portfolio.PortfolioId` via `collateral.PortfolioKey = portfolio.portfolio_id_thesis` |
| `PortfolioName` | `collateral.PortfolioName` |
| `OwnershipId` | `collateral.OwnershipKey` |
| `OwnershipPercentage` | `collateral.OwnershipPercentage` |
| `CollateralId` | populated from `collateral.CollateralKey` (ThesisCollateralID) — see Known Differences |
| `CollateralREStatus` | `collateral.RealEstateCollateralStatus` (Greek text preserved) |
| `CollateralREStatusChangeDate` | `collateral.CollateralREStatusChangeDate` |
| `collateral_status_en` | derived from `RealEstateCollateralStatus` (active/inactive) |
| `CollateralStatusId` | reconstructed from `collateral.CollateralStatus` (Ενεργή->11157, Ανενεργή->11212) |
| `OwnershipStatusId` | reconstructed from `collateral.OwnershipStatus` (Active->11006, Inactive->11009) |
| `AssetAmount` | `collateral.CollateralAmount` (on-prem AssetAmount == CollateralAmount) |
| `DateApplied` | `collateral.AppliedDate` |
| `ReferenceDate` | `run_date` |
| `ETLLoadingDate` | `current_date()` |

In [0]:
import sys
from datetime import datetime

dbutils.widgets.removeAll()
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD, empty=latest)")

sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import functions as F, Window

env = aws_utils.get_env_prefix()
src_catalog = f"`{env}-cdh`"

run_date = dbutils.widgets.get("run_date").strip()
if not run_date:
    run_date = str(
        spark.table(f"{src_catalog}.primaryactive.collateral")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

try:
    datetime.strptime(run_date, "%Y-%m-%d")
except ValueError:
    raise ValueError(f"run_date '{run_date}' must be YYYY-MM-DD")

run_date_lit = F.lit(run_date).cast("date")

TARGET_TABLE = f"{env}-cdh-delivery.it.CollateralsREAll"
sql_target = ".".join(f"`{p.strip('`')}`" for p in TARGET_TABLE.split("."))

print(f"env={env}  run_date={run_date}  target={TARGET_TABLE}")

In [0]:
# ==================== 1. Real-Estate collaterals ====================
# Category 'Επί ακινήτων' == on-prem CollateralTypeDescription 'Real Estate'.
# One row per collateral x property/ownership x agreement, matching on-prem grain.
collateral_re = (
    spark.table(f"{src_catalog}.primaryactive.collateral")
    .filter(
        (F.col("BusinessDate") == run_date_lit)
        & (F.col("Category") == "Επί ακινήτων")
        & F.col("RealEstateKey").isNotNull()
    )
)

# ==================== 2. Portfolio (on-prem DWH PortfolioId) ====================
pf_lookup = (
    spark.table(f"{src_catalog}.primaryactive.portfolio")
    .filter((F.col("BusinessDate") == run_date_lit) & F.col("portfolio_id_thesis").isNotNull())
    .select(
        F.col("portfolio_id_thesis").cast("string").alias("pf_thesis_id"),
        F.col("PortfolioId").alias("pf_PortfolioId"),
    )
    .dropDuplicates(["pf_thesis_id"])
)

# ==================== 3. Property key with CH-style prefix ====================
# On-prem PropertyKey = Thesis property asset key (e.g. CH230780) = realestatebuilding.RealEstateCepalKey
reb = (
    spark.table(f"{src_catalog}.primaryactive.realestatebuilding")
    .filter(F.col("BusinessDate") == run_date_lit)
    .select(
        F.col("RealEstateKey").cast("string").alias("reb_key"),
        F.col("RealEstateCepalKey").alias("reb_PropertyKey"),
    )
    .dropDuplicates(["reb_key"])
)

# ==================== 4. Property owner CDI (dparty via Thesis party key) ====================
owner_party = (
    spark.table(f"{src_catalog}.derivativeactive.dparty")
    .filter((F.col("BusinessDate") == run_date_lit) & F.col("PartyKeyThesis").isNotNull())
    .select(
        F.col("PartyKeyThesis").cast("string").alias("op_thesis_key"),
        F.col("CDI").alias("op_CDI"),
    )
    .dropDuplicates(["op_thesis_key"])
)

# ==================== 5. Agreement main-beneficiary CDI (primaryCDI) ====================
# On-prem AssetXPartyRltp type 2018 (Agreement-Main Beneficiary Rltp) == exposure 'Primary Owner'.
primary_cdi = (
    spark.table(f"{src_catalog}.primaryactive.exposure")
    .filter(
        (F.col("BusinessDate") == run_date_lit)
        & (F.col("PartyRelationshipType") == "Primary Owner")
        & F.col("AgreementKey").isNotNull()
    )
    .select(
        F.col("AgreementKey").alias("pc_AgreementKey"),
        F.col("CDI").alias("pc_CDI"),
    )
    .dropDuplicates(["pc_AgreementKey"])
)

In [0]:
# on-prem active set for collateral_status_en
ACTIVE_RE = ["ΕΝΕΡΓΗ", "Ενεργή", "Ενεργη", "ενεργή", "ενεργη", "ΟΥΣΙΑΣ"]

df = (
    collateral_re
    .join(pf_lookup, F.col("PortfolioKey").cast("string") == F.col("pf_thesis_id"), "left")
    .join(reb, F.col("RealEstateKey").cast("string") == F.col("reb_key"), "left")
    .join(owner_party, F.col("OwnerPartyKey").cast("string") == F.col("op_thesis_key"), "left")
    # INNER join mirrors on-prem AssetXPartyRltp type 2018 (agreement main beneficiary):
    # collaterals whose agreement has no Primary Owner are dropped, as on-prem.
    .join(primary_cdi, F.col("AgreementKey") == F.col("pc_AgreementKey"), "inner")
)

no_cdi_fallback = F.concat(F.lit("NoCDI - PropertyOwnerId = "), F.col("OwnershipKey").cast("string"))

result = df.select(
    F.col("reb_PropertyKey").cast("string").alias("PropertyKey"),
    F.coalesce(F.col("op_CDI"), no_cdi_fallback).cast("string").alias("CDI"),
    F.col("AgreementKey").cast("string").alias("AgreementNumber"),
    F.col("InitialLegalOrder").cast("int").alias("IntialLegalOrder"),
    F.col("CollateralCurrency").cast("string").alias("Currency"),
    # Convert to EUR using the cloud UDF (EUR passes through unchanged).
    F.expr(
        f"{src_catalog}.derivative.converttoeur(CollateralCurrency, "
        f"CAST(CollateralAmount AS decimal(19,2)), CAST('{run_date}' AS date))"
    ).cast("decimal(19,2)").alias("CollateralAmountEuro"),
    F.col("CollateralAmount").cast("decimal(19,2)").alias("CollateralAmount"),
    F.col("pc_CDI").cast("string").alias("primaryCDI"),
    F.col("CollateralOldKey").cast("string").alias("CollateralKey"),
    F.col("CollateralKey").cast("string").alias("ThesisCollateralID"),
    F.col("pf_PortfolioId").cast("int").alias("PortfolioId"),
    F.col("PortfolioName").cast("string").alias("PortfolioName"),
    F.col("OwnershipKey").cast("int").alias("OwnershipId"),
    F.col("OwnershipPercentage").cast("decimal(18,6)").alias("OwnershipPercentage"),
    F.col("CollateralKey").cast("string").alias("CollateralId"),
    F.col("RealEstateCollateralStatus").cast("string").alias("CollateralREStatus"),
    F.col("CollateralREStatusChangeDate").cast("date").alias("CollateralREStatusChangeDate"),
    F.when(F.col("RealEstateCollateralStatus").isin(ACTIVE_RE), F.lit("active"))
     .otherwise(F.lit("inactive")).alias("collateral_status_en"),
    F.when(F.col("CollateralStatus") == "Ενεργή", F.lit(11157))
     .when(F.col("CollateralStatus") == "Ανενεργή", F.lit(11212))
     .otherwise(F.lit(None)).cast("int").alias("CollateralStatusId"),
    F.when(F.col("OwnershipStatus") == "Active", F.lit(11006))
     .when(F.col("OwnershipStatus") == "Inactive", F.lit(11009))
     .otherwise(F.lit(None)).cast("int").alias("OwnershipStatusId"),
    F.col("CollateralAmount").cast("decimal(19,2)").alias("AssetAmount"),
    F.col("AppliedDate").cast("date").alias("DateApplied"),
    run_date_lit.alias("ReferenceDate"),
    F.current_date().alias("ETLLoadingDate"),
).distinct()

row_count = result.count()
assert row_count > 0, f"0 rows for run_date={run_date} -- aborting write"
print(f"Records to write: {row_count:,}")

## Known differences vs on-prem (document for the external consumer)

The table preserves the on-prem schema and values wherever the cloud model allows. The following are the accepted or reconstructed differences:

1. **`CollateralStatusId` / `OwnershipStatusId` (reconstructed).** On-prem stores DWH `Classification` numeric ids. The cloud curated layer keeps only the status *text*, so the numeric ids are reconstructed from a fixed, tiny domain: `CollateralStatusId` = 11157 (`Ενεργή`) / 11212 (`Ανενεργή`); `OwnershipStatusId` = 11006 (`Active`) / 11009 (`Inactive`). Any future status value not in this map yields NULL.

2. **`CollateralId` = `ThesisCollateralID`.** On-prem exposes the DWH internal numeric `Collateral.CollateralId`, which does not exist in the cloud model. By decision it is populated from the stable Thesis collateral id (`primaryactive.collateral.CollateralKey`, same value as `ThesisCollateralID`).

3. **`CollateralAmountEuro`.** Computed with the cloud UDF `{env}-cdh.derivative.converttoeur(currency, amount, rate_date)` using `run_date` as the rate date. EUR passes through unchanged; other currencies (~1.4K rows) are converted to EUR. This differs from on-prem, where `bi.convertToEuro` failed to resolve an FX rate and left non-EUR amounts **NULL** — the cloud version is an improvement that now populates those rows.

4. **`primaryCDI` uses an INNER join.** On-prem requires the agreement main-beneficiary relationship (`AssetXPartyRltp` type 2018). This is mirrored with an INNER join to `exposure` Primary Owner, so collaterals whose agreement has no Primary Owner are dropped, matching on-prem.

5. **Owner `CDI` fallback.** When the property owner party has no CDI (~430 rows), the on-prem `NoCDI - PropertyOwnerId = <OwnershipId>` fallback string is reproduced.

6. **Row-count parity (verified).** On 2026-09-01 the notebook produces 349,242 rows vs on-prem 349,232 — a 10-row difference (~0.003%). Distinct collaterals match by status (active 214,194 vs 214,191; inactive 28,835 vs 28,835) and the grain (avg ~1.43 rows / collateral) is identical. Alignment relies on two behaviours that mirror the on-prem SP: (a) the INNER join to `exposure` Primary Owner drops collaterals whose agreement has no main beneficiary (e.g. `Πώληση σε νέο χαρτ.` collaterals moved to another portfolio), matching the on-prem `AssetXPartyRltp` type 2018 inner join; (b) `dropDuplicates(["reb_key"])` on `realestatebuilding` avoids grain fan-out and is safe because `RealEstateKey -> RealEstateCepalKey` is 1:1. The residual ~10 rows are SCD/reference-date timing noise.

7. **`ETLLoadingDate` = `current_date()`** and **`ReferenceDate` = `run_date`** replace the on-prem `SourceExtraction` (Thesis) reference date.

In [0]:
(
    result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(sql_target)
)

print(f"Written {row_count:,} rows to {TARGET_TABLE}  (ReferenceDate={run_date})")